In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 120

# --- DuckDB setup with memory safety ---
con = duckdb.connect()
con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 4")

# --- Register all datasets as views ---
COSTS_PATH = 'data/costs/costs.parquet'
PRICES_PATH = 'data/prices/prices.parquet'
SIM_DAILY_PATH = 'data/sim_daily/*.parquet'
SIM_MONTHLY_PATH = 'data/sim_monthly/*.parquet'

con.execute(f"CREATE VIEW costs AS SELECT * FROM read_parquet('{COSTS_PATH}')")
con.execute(f"""
    CREATE VIEW prices AS
    SELECT *, STRFTIME(DATETIME, '%Y-%m') AS MONTH, YEAR(DATETIME) AS YEAR
    FROM read_parquet('{PRICES_PATH}')
""")
con.execute(f"""
    CREATE VIEW sim_daily AS
    SELECT *, YEAR(DATETIME) AS YEAR, MONTH(DATETIME) AS MO, DAY(DATETIME) AS DAY
    FROM read_parquet('{SIM_DAILY_PATH}')
""")

# sim_monthly: same structure as sim_daily but PSM instead of PSD
SIM_MONTHLY_AVAILABLE = os.path.exists('data/sim_monthly')
if SIM_MONTHLY_AVAILABLE and len(os.listdir('data/sim_monthly')) > 0:
    con.execute(f"""
        CREATE VIEW sim_monthly AS
        SELECT *, YEAR(DATETIME) AS YEAR, MONTH(DATETIME) AS MO
        FROM read_parquet('{SIM_MONTHLY_PATH}')
    """)
    print("View sim_monthly created.")
else:
    SIM_MONTHLY_AVAILABLE = False
    print("sim_monthly NOT available — sections using PSM will be skipped.")

print("Views created: costs, prices, sim_daily" + (", sim_monthly" if SIM_MONTHLY_AVAILABLE else ""))

## Section 1 — Data Inventory

Schema, row counts, and EID overlap across datasets. The **costs** table (927 EIDs) defines the opportunity universe.

In [ ]:
# --- Schema inspection ---
for view_name in ['costs', 'prices', 'sim_daily']:
    print(f"\n{'='*60}")
    print(f"  {view_name.upper()}")
    print(f"{'='*60}")
    schema = con.execute(f"DESCRIBE {view_name}").fetchdf()
    print(schema.to_string(index=False))
    count = con.execute(f"SELECT COUNT(*) AS n FROM {view_name}").fetchone()[0]
    print(f"\nTotal rows: {count:,}")

if SIM_MONTHLY_AVAILABLE:
    print(f"\n{'='*60}")
    print(f"  SIM_MONTHLY")
    print(f"{'='*60}")
    schema = con.execute("DESCRIBE sim_monthly").fetchdf()
    print(schema.to_string(index=False))
    count = con.execute("SELECT COUNT(*) AS n FROM sim_monthly").fetchone()[0]
    print(f"\nTotal rows: {count:,}")
else:
    print("\n[SKIP] sim_monthly not available")

In [ ]:
# --- EID overlap analysis ---
eid_costs = con.execute("SELECT DISTINCT EID FROM costs").fetchdf()['EID']
eid_prices = con.execute("SELECT DISTINCT EID FROM prices").fetchdf()['EID']
eid_sim_daily = con.execute("SELECT DISTINCT EID FROM sim_daily").fetchdf()['EID']

print(f"Distinct EIDs in costs:      {len(eid_costs)}")
print(f"Distinct EIDs in prices:     {len(eid_prices)}")
print(f"Distinct EIDs in sim_daily:  {len(eid_sim_daily)}")

# Overlap with costs (the opportunity universe)
costs_set = set(eid_costs)
prices_set = set(eid_prices)
sim_daily_set = set(eid_sim_daily)

print(f"\nCosts EIDs also in prices:     {len(costs_set & prices_set)} / {len(costs_set)}")
print(f"Costs EIDs also in sim_daily:  {len(costs_set & sim_daily_set)} / {len(costs_set)}")
print(f"Costs EIDs missing prices:     {len(costs_set - prices_set)}")
print(f"Costs EIDs missing sim_daily:  {len(costs_set - sim_daily_set)}")

# Month coverage
months_costs = con.execute("SELECT DISTINCT MONTH FROM costs ORDER BY MONTH").fetchdf()
months_prices = con.execute("SELECT DISTINCT MONTH FROM prices ORDER BY MONTH").fetchdf()
print(f"\nMonth range in costs:  {months_costs['MONTH'].iloc[0]} to {months_costs['MONTH'].iloc[-1]} ({len(months_costs)} months)")
print(f"Month range in prices: {months_prices['MONTH'].iloc[0]} to {months_prices['MONTH'].iloc[-1]} ({len(months_prices)} months)")

## Section 2 — Target Construction

**PROFIT = |PR| - |C|** (updated jury formula).

- `PR = ABS(SUM(PRICEREALIZED))` per (EID, MONTH, PEAKID)
- `C = ABS(cost)` from costs table
- `TARGET = 1` if PROFIT > 0, else 0

Anchor on **costs** table (927 EIDs = opportunity universe).

In [ ]:
# --- Build target table ---
target_df = con.execute("""
    WITH monthly_pr AS (
        SELECT EID, STRFTIME(DATETIME, '%Y-%m') AS MONTH, PEAKID,
               SUM(PRICEREALIZED) AS PR_signed,
               ABS(SUM(PRICEREALIZED)) AS PR
        FROM prices
        GROUP BY EID, MONTH, PEAKID
    )
    SELECT c.EID, c.MONTH, c.PEAKID,
           ABS(c.C) AS C,
           COALESCE(pr.PR, 0) AS PR,
           COALESCE(pr.PR_signed, 0) AS PR_signed,
           COALESCE(pr.PR, 0) - ABS(c.C) AS PROFIT,
           CASE WHEN COALESCE(pr.PR, 0) - ABS(c.C) > 0 THEN 1 ELSE 0 END AS TARGET
    FROM costs c
    LEFT JOIN monthly_pr pr
        ON c.EID = pr.EID AND c.MONTH = pr.MONTH AND c.PEAKID = pr.PEAKID
    ORDER BY c.MONTH, c.EID, c.PEAKID
""").fetchdf()

print(f"Target table shape: {target_df.shape}")
print(f"\nTarget distribution:")
print(target_df['TARGET'].value_counts())
print(f"\nProfitability rate (new formula |PR|-|C|): {target_df['TARGET'].mean()*100:.2f}%")

# Compare with old formula for reference
old_profit = target_df['PR_signed'] - target_df['C']
old_target = (old_profit > 0).astype(int)
print(f"Profitability rate (old formula  PR-C):   {old_target.mean()*100:.2f}%")

print(f"\n--- Target table sample ---")
target_df.head(10)

In [ ]:
# --- PROFIT distribution visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PROFIT histogram
axes[0].hist(target_df['PROFIT'], bins=80, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', label='Break-even')
axes[0].set_xlabel('PROFIT = |PR| - |C|')
axes[0].set_ylabel('Count')
axes[0].set_title('Profit Distribution (Updated Formula)')
axes[0].legend()

# By PEAKID
for pid, label in [(0, 'OFF-Peak'), (1, 'ON-Peak')]:
    subset = target_df[target_df['PEAKID'] == pid]
    rate = subset['TARGET'].mean() * 100
    axes[1].bar(label, rate, color='steelblue' if pid == 0 else 'coral')
    axes[1].text(label, rate + 0.5, f'{rate:.1f}%', ha='center', fontweight='bold')

axes[1].set_ylabel('Profitability Rate (%)')
axes[1].set_title('Profitability Rate by Peak Type')

plt.tight_layout()
plt.show()

## Section 3 — Anti-Leakage Time Alignment

Decision is made on the **7th of month M** for **target month M+1**.

| Data Source | Available Window | Key |
|-------------|-----------------|-----|
| sim_daily | Days 1-7 of month **M** | `DECISION_MONTH = M` |
| sim_monthly | All hours of month **M+1** | `TARGET_MONTH = M+1` (allowed) |
| prices | Up to day 7 of month **M** | Historical features only |
| costs | Month **M** cost | Lag feature (C for M+1 is FORBIDDEN) |

We create a `DECISION_MONTH` column = TARGET_MONTH minus 1 month, to align features correctly.

In [ ]:
# --- Add DECISION_MONTH to target table ---
# DECISION_MONTH M = TARGET_MONTH M+1 minus 1 month
# i.e., for TARGET_MONTH '2020-02', DECISION_MONTH = '2020-01'
target_df['DECISION_MONTH'] = pd.to_datetime(target_df['MONTH']) - pd.DateOffset(months=1)
target_df['DECISION_MONTH'] = target_df['DECISION_MONTH'].dt.strftime('%Y-%m')

# First TARGET_MONTH has no decision month with available sim_daily features
print(f"TARGET_MONTH range: {target_df['MONTH'].min()} to {target_df['MONTH'].max()}")
print(f"DECISION_MONTH range: {target_df['DECISION_MONTH'].min()} to {target_df['DECISION_MONTH'].max()}")
print(f"\nAlignment table sample:")
target_df[['EID', 'MONTH', 'DECISION_MONTH', 'PEAKID', 'C', 'PR', 'PROFIT', 'TARGET']].head(10)

## Section 4 — Feature Engineering from sim_daily

Extract features from **days 1-7 of DECISION_MONTH M** per `(EID, PEAKID)`.

Key design choices:
- **Magnitudes** (`ABS()`) are primary features since `PROFIT = |PR| - |C|`
- **Scenario-specific** features: S1 differs from S2/S3 (r=0.22 vs r=0.87)
- **Trend** features: early (days 1-3) vs late (days 4-7)
- **Year-by-year** processing to stay within 2GB DuckDB memory limit

In [ ]:
# --- sim_daily feature extraction (year-by-year) ---
SIM_DAILY_FEATURE_QUERY = """
    SELECT EID,
           STRFTIME(DATETIME, '%Y-%m') AS DECISION_MONTH,
           PEAKID,
           -- Congestion signal (magnitude = key predictor of |PR|)
           SUM(CASE WHEN PSD != 0 THEN 1 ELSE 0 END) AS psd_nonzero_count,
           AVG(CASE WHEN PSD != 0 THEN ABS(PSD) END) AS psd_abs_nonzero_mean,
           STDDEV(CASE WHEN PSD != 0 THEN ABS(PSD) END) AS psd_abs_nonzero_std,
           SUM(ABS(PSD)) AS psd_abs_sum,
           -- Signed PSD for directional analysis
           AVG(CASE WHEN PSD != 0 THEN PSD END) AS psd_signed_mean,
           MAX(ABS(PSD)) AS psd_abs_max,
           -- Activation level
           AVG(ACTIVATIONLEVEL) AS activation_mean,
           MAX(ACTIVATIONLEVEL) AS activation_max,
           SUM(CASE WHEN ACTIVATIONLEVEL > 0 THEN 1 ELSE 0 END) AS activation_nonzero_count,
           -- Impact magnitudes
           AVG(ABS(WINDIMPACT)) AS wind_abs_mean,
           AVG(ABS(SOLARIMPACT)) AS solar_abs_mean,
           AVG(ABS(HYDROIMPACT)) AS hydro_abs_mean,
           AVG(ABS(NONRENEWBALIMPACT)) AS nonrenew_abs_mean,
           AVG(ABS(EXTERNALIMPACT)) AS external_abs_mean,
           AVG(ABS(LOADIMPACT)) AS load_abs_mean,
           AVG(ABS(TRANSMISSIONOUTAGEIMPACT)) AS transoutage_abs_mean,
           -- Scenario-specific PSD (S1 differs from S2/S3)
           AVG(CASE WHEN SCENARIOID = 1 THEN ABS(PSD) END) AS psd_abs_s1_mean,
           AVG(CASE WHEN SCENARIOID IN (2, 3) THEN ABS(PSD) END) AS psd_abs_s23_mean,
           -- Inter-scenario spread (uncertainty signal)
           STDDEV(CASE WHEN PSD != 0 THEN PSD END) AS psd_scenario_spread,
           -- Trend: early vs late in 7-day window
           AVG(CASE WHEN DAY(DATETIME) <= 3 THEN ABS(PSD) END) AS psd_abs_early,
           AVG(CASE WHEN DAY(DATETIME) BETWEEN 4 AND 7 THEN ABS(PSD) END) AS psd_abs_late
    FROM read_parquet('data/sim_daily/sim_daily_{year}.parquet')
    WHERE DAY(DATETIME) BETWEEN 1 AND 7
    GROUP BY EID, DECISION_MONTH, PEAKID
"""

sim_daily_features_list = []
for year in [2020, 2021, 2022, 2023]:
    print(f"Processing sim_daily {year}...", end=" ")
    query = SIM_DAILY_FEATURE_QUERY.replace('{year}', str(year))
    df_year = con.execute(query).fetchdf()
    sim_daily_features_list.append(df_year)
    print(f"{len(df_year):,} rows")

sim_daily_features = pd.concat(sim_daily_features_list, ignore_index=True)
print(f"\nTotal sim_daily features: {sim_daily_features.shape}")
print(f"Columns: {list(sim_daily_features.columns)}")
sim_daily_features.head()

## Section 5 — Feature Engineering from sim_monthly

Extract features from **TARGET_MONTH M+1** using PSM (allowed per anti-leakage rules).

Same feature pattern as sim_daily but:
- No day filter (all hours of target month available)
- PSM instead of PSD
- Skipped gracefully if sim_monthly data not available

In [ ]:
# --- sim_monthly feature extraction ---
if SIM_MONTHLY_AVAILABLE:
    SIM_MONTHLY_FEATURE_QUERY = """
        SELECT EID,
               STRFTIME(DATETIME, '%Y-%m') AS TARGET_MONTH,
               PEAKID,
               -- PSM congestion signal for target month
               SUM(CASE WHEN PSM != 0 THEN 1 ELSE 0 END) AS psm_nonzero_count,
               AVG(CASE WHEN PSM != 0 THEN ABS(PSM) END) AS psm_abs_nonzero_mean,
               STDDEV(CASE WHEN PSM != 0 THEN ABS(PSM) END) AS psm_abs_nonzero_std,
               SUM(ABS(PSM)) AS psm_abs_sum,
               AVG(CASE WHEN PSM != 0 THEN PSM END) AS psm_signed_mean,
               MAX(ABS(PSM)) AS psm_abs_max,
               -- Activation level
               AVG(ACTIVATIONLEVEL) AS psm_activation_mean,
               MAX(ACTIVATIONLEVEL) AS psm_activation_max,
               -- Impact magnitudes
               AVG(ABS(WINDIMPACT)) AS psm_wind_abs_mean,
               AVG(ABS(SOLARIMPACT)) AS psm_solar_abs_mean,
               AVG(ABS(HYDROIMPACT)) AS psm_hydro_abs_mean,
               AVG(ABS(NONRENEWBALIMPACT)) AS psm_nonrenew_abs_mean,
               AVG(ABS(EXTERNALIMPACT)) AS psm_external_abs_mean,
               -- Scenario-specific
               AVG(CASE WHEN SCENARIOID = 1 THEN ABS(PSM) END) AS psm_abs_s1_mean,
               AVG(CASE WHEN SCENARIOID IN (2, 3) THEN ABS(PSM) END) AS psm_abs_s23_mean,
               STDDEV(CASE WHEN PSM != 0 THEN PSM END) AS psm_scenario_spread
        FROM read_parquet('data/sim_monthly/sim_monthly_{year}.parquet')
        GROUP BY EID, TARGET_MONTH, PEAKID
    """

    sim_monthly_features_list = []
    for year in [2020, 2021, 2022, 2023]:
        print(f"Processing sim_monthly {year}...", end=" ")
        query = SIM_MONTHLY_FEATURE_QUERY.replace('{year}', str(year))
        df_year = con.execute(query).fetchdf()
        sim_monthly_features_list.append(df_year)
        print(f"{len(df_year):,} rows")

    sim_monthly_features = pd.concat(sim_monthly_features_list, ignore_index=True)
    print(f"\nTotal sim_monthly features: {sim_monthly_features.shape}")
else:
    sim_monthly_features = None
    print("[SKIP] sim_monthly not available — PSM features will be absent from master dataset.")

## Section 6 — Historical Features from Prices & Costs

Lagged and rolling features using **absolute values** (aligned with updated profit formula).

For each opportunity `(EID, TARGET_MONTH M+1, PEAKID)`:
- `|PR|` for months M, M-1, M-2 (lagged realized prices)
- `|C|` for month M (cost lag)
- Rolling averages and profit history
- Count of historically profitable months

In [ ]:
# --- Historical features from target_df itself ---
# target_df already has monthly PR, C, PROFIT per (EID, MONTH, PEAKID)
# We compute lagged features using pandas window operations

# Sort for proper lag computation
target_sorted = target_df.sort_values(['EID', 'PEAKID', 'MONTH']).copy()

# Group by (EID, PEAKID) and compute lags
grouped = target_sorted.groupby(['EID', 'PEAKID'])

# Lag features (looking backwards from TARGET_MONTH)
target_sorted['pr_lag1'] = grouped['PR'].shift(1)       # |PR| for month M (1 month before target)
target_sorted['pr_lag2'] = grouped['PR'].shift(2)       # |PR| for month M-1
target_sorted['pr_lag3'] = grouped['PR'].shift(3)       # |PR| for month M-2
target_sorted['c_lag1'] = grouped['C'].shift(1)         # |C| for month M
target_sorted['profit_lag1'] = grouped['PROFIT'].shift(1)  # PROFIT for month M
target_sorted['profit_lag2'] = grouped['PROFIT'].shift(2)
target_sorted['target_lag1'] = grouped['TARGET'].shift(1)  # Was previous month profitable?

# Rolling features (3-month window, shifted by 1 to avoid leakage)
target_sorted['pr_rolling3_mean'] = grouped['PR'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)
target_sorted['pr_rolling3_std'] = grouped['PR'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).std()
)
target_sorted['profit_rolling3_mean'] = grouped['PROFIT'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)
target_sorted['profitable_count_3m'] = grouped['TARGET'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).sum()
)

hist_feature_cols = ['pr_lag1', 'pr_lag2', 'pr_lag3', 'c_lag1',
                     'profit_lag1', 'profit_lag2', 'target_lag1',
                     'pr_rolling3_mean', 'pr_rolling3_std',
                     'profit_rolling3_mean', 'profitable_count_3m']

print("Historical features added:")
for col in hist_feature_cols:
    non_null = target_sorted[col].notna().sum()
    print(f"  {col}: {non_null:,} non-null values ({non_null/len(target_sorted)*100:.1f}%)")

target_with_hist = target_sorted.copy()

## Section 7 — Master Merge

Join all feature tables with proper time-shift alignment:
- **target_with_hist** `(EID, MONTH=M+1, PEAKID)` — target + historical features
- **sim_daily_features** `(EID, DECISION_MONTH=M, PEAKID)` — days 1-7 of M
- **sim_monthly_features** `(EID, TARGET_MONTH=M+1, PEAKID)` — full month M+1
- **Calendar features**: month_of_year, season

In [ ]:
# --- Master merge ---
master_df = target_with_hist.copy()

# Join sim_daily features on DECISION_MONTH (= TARGET_MONTH - 1)
master_df = master_df.merge(
    sim_daily_features,
    left_on=['EID', 'DECISION_MONTH', 'PEAKID'],
    right_on=['EID', 'DECISION_MONTH', 'PEAKID'],
    how='left',
    suffixes=('', '_sd')
)
print(f"After sim_daily join: {master_df.shape}")

# Join sim_monthly features on TARGET_MONTH
if sim_monthly_features is not None:
    master_df = master_df.merge(
        sim_monthly_features,
        left_on=['EID', 'MONTH', 'PEAKID'],
        right_on=['EID', 'TARGET_MONTH', 'PEAKID'],
        how='left',
        suffixes=('', '_sm')
    )
    master_df.drop(columns=['TARGET_MONTH'], inplace=True, errors='ignore')
    print(f"After sim_monthly join: {master_df.shape}")
else:
    print("[SKIP] sim_monthly features not available")

# --- Calendar features ---
master_df['month_of_year'] = pd.to_datetime(master_df['MONTH']).dt.month
master_df['season'] = master_df['month_of_year'].map({
    12: 'winter', 1: 'winter', 2: 'winter',
    3: 'spring', 4: 'spring', 5: 'spring',
    6: 'summer', 7: 'summer', 8: 'summer',
    9: 'fall', 10: 'fall', 11: 'fall'
})

# Fill NaN sim features with 0 (sparse data: missing = no congestion signal)
sim_cols = [c for c in master_df.columns if c.startswith(('psd_', 'psm_', 'activation', 'wind_', 'solar_',
            'hydro_', 'nonrenew_', 'external_', 'load_', 'transoutage_'))]
master_df[sim_cols] = master_df[sim_cols].fillna(0)

print(f"\nFinal master dataset: {master_df.shape}")
print(f"Columns ({len(master_df.columns)}): {list(master_df.columns)}")

## Section 8 — Data Quality Check

- Missing values per feature
- Feature-target correlations (top predictors)
- Class balance confirmation
- Sanity check: features from M should predict TARGET for M+1

In [ ]:
# --- Missing values ---
missing = master_df.isnull().sum()
missing_pct = (missing / len(master_df) * 100).round(1)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_report = missing_report[missing_report['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print("Features with missing values:")
print(missing_report.to_string())

# --- Feature-target correlations ---
numeric_cols = master_df.select_dtypes(include=[np.number]).columns
numeric_cols = [c for c in numeric_cols if c not in ['EID', 'PEAKID', 'TARGET', 'PR', 'PR_signed', 'C', 'PROFIT']]

correlations = master_df[numeric_cols + ['TARGET']].corr()['TARGET'].drop('TARGET').abs().sort_values(ascending=False)

print(f"\n--- Top 20 features correlated with TARGET (|PR|-|C| > 0) ---")
print(correlations.head(20).to_string())

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top correlations bar chart
top_corr = correlations.head(15)
top_corr.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('|Correlation| with TARGET')
axes[0].set_title('Top 15 Feature-Target Correlations')
axes[0].invert_yaxis()

# Class balance by month
monthly_balance = master_df.groupby('MONTH')['TARGET'].mean() * 100
monthly_balance.plot(ax=axes[1], marker='o', color='coral')
axes[1].set_ylabel('Profitability Rate (%)')
axes[1].set_title('Monthly Profitability Rate')
axes[1].axhline(y=master_df['TARGET'].mean()*100, color='gray', linestyle='--', alpha=0.5, label='Overall avg')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Section 9 — Export Master Dataset

Save the final master dataset as Parquet for the modeling pipeline.

Output: `data/master_dataset.parquet`

In [ ]:
# --- Export ---
# Drop rows where lag features are fully unavailable (first month per EID)
master_export = master_df.dropna(subset=['pr_lag1']).copy()
print(f"Rows after dropping first month (no lag features): {len(master_export):,} (dropped {len(master_df) - len(master_export):,})")

# Encode season as numeric for modeling
season_map = {'winter': 0, 'spring': 1, 'summer': 2, 'fall': 3}
master_export['season_encoded'] = master_export['season'].map(season_map)

# Save
output_path = 'data/master_dataset.parquet'
master_export.to_parquet(output_path, index=False)
print(f"\nExported to {output_path}")
print(f"Final shape: {master_export.shape}")
print(f"Target rate: {master_export['TARGET'].mean()*100:.2f}%")

# --- Summary ---
print(f"\n{'='*60}")
print(f"  MASTER DATASET SUMMARY")
print(f"{'='*60}")
print(f"  Rows: {len(master_export):,}")
print(f"  Columns: {len(master_export.columns)}")
print(f"  Target (profitable): {master_export['TARGET'].sum():,} ({master_export['TARGET'].mean()*100:.1f}%)")
print(f"  Unique EIDs: {master_export['EID'].nunique()}")
print(f"  Month range: {master_export['MONTH'].min()} to {master_export['MONTH'].max()}")
print(f"  sim_monthly features: {'Yes' if sim_monthly_features is not None else 'No (placeholder)'}")

feature_cols = [c for c in master_export.columns if c not in
                ['EID', 'MONTH', 'DECISION_MONTH', 'PEAKID', 'C', 'PR', 'PR_signed', 'PROFIT', 'TARGET', 'season']]
print(f"  Feature columns: {len(feature_cols)}")
print(f"\n  Feature list:")
for col in feature_cols:
    print(f"    - {col}")

# Close DuckDB connection
con.close()
print(f"\nDuckDB connection closed.")